# oncoct — Colab A100 driver

Runs the GPU-heavy stages (detection, MedSAM2, malignancy fine-tune) on an ephemeral A100.
**Sessions die and lose state** — mount Drive, point weight caches at it, and checkpoint often.
See `plan.md` §5 (compute plan).

**Round 1 is the DETERMINISTIC pipeline, not the live LLM agent** — cell 4 runs
`scripts/run_pipeline.py` (fixed stage order), which is the Round-1 stand-in for the agent.

In [ ]:
# 1. Mount Drive for persistent weight caches + checkpoints
from google.colab import drive
drive.mount('/content/drive')
import os
os.environ['TOTALSEG_WEIGHTS_PATH'] = '/content/drive/MyDrive/oncoct/weights/totalsegmentator'

In [ ]:
# 2. Clone + install PRIMARY env deps. MedSAM2 needs its OWN env (torch 2.5.1/cu124) —
#    see envs/medsam2.yml; run it as a separate process via scripts/medsam2_worker.py.
!git clone <REPO_URL> oncoct && cd oncoct && pip install -e '.[detect,labels,radiomics]' matplotlib scikit-learn

In [ ]:
# 3. Seed weights to Drive (once) — survives session resets
!cd oncoct && python scripts/seed_weights.py --to /content/drive/MyDrive/oncoct/weights

In [ ]:
# 4. End-to-end on example studies (Round 1 = DETERMINISTIC driver, NOT the LLM agent).
#    ingest->detect->segment->measure->attribute->classify->report; writes
#    results/reports/*.json + *.txt and results/overlays/*.png
!cd oncoct && python scripts/run_pipeline.py \
    --series data/luna16/subset0 --config configs/pipeline.yaml --out results/

In [ ]:
# 5. (optional) Light malignancy fine-tune — checkpoints to Drive every few minutes
!cd oncoct && python scripts/finetune_malignancy.py --epochs 30 \
    --ckpt /content/drive/MyDrive/oncoct/weights/malignancy

In [ ]:
# 6. Benchmarks (evidence the pieces are real): FROC on detection, Dice on segmentation.
#    Detections must be written to a WORLD-coord CSV first — eval/froc_luna16.py provides
#    write_predictions_csv(); pass the excluded set for a correct CPM.
#    NOTE: adjust --annotations-excluded to whatever the official LUNA16 evaluation set is
#    actually named (filename varies — see BRIEF §4).
!cd oncoct && python eval/froc_luna16.py --predictions results/metrics/preds.csv \
    --annotations data/luna16/annotations.csv \
    --annotations-excluded data/luna16/annotations_excluded.csv \
    --output results/metrics/froc/